# Otimização do Fluxo de Checkout: Teste A/B no E-Commerce

## 1. Contexto do Negócio

A equipe de produto identificou uma alta taxa de abandono no fluxo de checkout da plataforma mobile. Análises de navegação indicaram que a complexidade da última etapa de confirmação do pedido causava atrito nos usuários, gerando um gargalo no funil de conversão.

## 2. Formulação de Hipóteses

### Problema

O formulário de finalização de compra atual exige preenchimento em múltiplas páginas com etapas redundantes, o que diminui a taxa de conversão final no checkout.

### Hipótese Formal

> **SE** simplificarmos o fluxo de checkout em uma única página (*One-Step Checkout*) e destacarmos o botão de ação principal,
> **ENTÃO** reduziremos o atrito cognitivo dos usuários,
> **PARA QUE** a taxa de conversão de compras (*Purchase Conversion Rate*) aumente em pelo menos **2% absolutos (25% relativo)** sem impactar negativamente a receita média por usuário (ARPU).

---

A diferença entre **MDE Absoluto** e **MDE Relativo** está na **unidade de medida** usada para definir o Efeito Mínimo Detectável (*Minimum Detectable Effect*). O MDE é a menor diferença na taxa que você deseja que seu teste seja estatisticamente capaz de detectar.

| Característica | MDE Absoluto | MDE Relativo |
| --- | --- | --- |
| **Definição** | A **diferença direta de pontos percentuais** entre o Grupo B e o Controle. | O **percentual de aumento ou queda em relação à taxa base** do Controle. |
| **Fórmula** | $\text{MDE}_{\text{Absoluto}} = p_{\text{Tratamento}} - p_{\text{Controle}}$ | $\text{MDE}_{\text{Relativo}} = \frac{p_{\text{Tratamento}} - p_{\text{Controle}}}{p_{\text{Controle}}}$ |
| **Como é comunicado** | *"Queremos detectar um aumento de 2 pontos percentuais."* | *"Queremos detectar uma elevação (lift) de 25%."* |

### Exemplo 

Taxa de conversão do Grupo Controle é **$8\%$** ($p_1 = 0{,}08$) e você quer testar um novo checkout:

**MDE Absoluto de $2\%$ ($0{,}02$):**
* Você quer que o teste detecte se a nova taxa vai para **$10\%$** ($8\% + 2\%$).

**MDE Relativo de $25\%$ ($0{,}25$):**
* Você quer que o teste detecte um ganho de $25\%$ *sobre os $8\%$ atuais*.
    - Cálculo: $8\% \times (1 + 0{,}25) =$ **$10\%$**.

### Quando usar cada um?

* **MDE Relativo** é o **mais usado no dia a dia por times de Produto e Negócio**, pois facilita a comparação do impacto do teste em relação ao tamanho atual da métrica (um ganho relativo de +20% é intuitivo para qualquer taxa base).
* **MDE Absoluto** é o valor que entra **diretamente nas equações estatísticas** e fórmulas do teste Z para dimensionar o desvio padrão e o erro amostral. Em nossa função, usamos o MDE relativo no parâmetro da função, porém ele foi convertido em absoluto internamente (`p2 - p1`).

### Definição de Métricas

* **Métrica Primária (Sucesso):** Taxa de Conversão de Compras (`Converteu` = 1 ou 0).
* **Métrica Secundária (Direcionadora):** Valor total da compra por usuário (`Valor gasto`).
* **Métricas Guardrail (Segurança):** Tempo gasto no site (`Tempo gasto`) e total de páginas visitadas (`N° de paginas visitadas`).

## 3. Dimensionamento Amostral (Análise de poder)

Para evitar os erros de **Tipo I ($\alpha$)** (falsos positivos) e **Tipo II ($\beta$)** (falsos negativos), realizamos o cálculo do tamanho de amostra necessário antes do lançamento.

Caso haja mais de uma variante tratando o controle ($A/B/n$), aplicamos a **Correção de Bonferroni** no nível de significância para mitigar a inflação do erro do Tipo I decorrente de comparações múltiplas:

$$\alpha_{corrigido} = \frac{\alpha}{k}$$

*onde $k$ é o número de comparações ativas contra o controle ($k = \text{variantes} - 1$).*



In [ ]:
import numpy as np
import pandas as pd 
import scipy.stats as stats

In [ ]:
def calcular_tamanho_amostra(baseline_conversion: float, mde_relative: float, alpha: float = 0.05, power: float = 0.80, n_variants: int = 2, alternative: str = "one-sided") -> dict:
    """Calcula o tamanho amostral necessário por grupo para um Teste A/B/n.

    Parâmetros:
    -----------
    baseline_conversion : float
        Taxa de conversão atual do grupo controle (ex: 0.08 para 8%).
    mde_relative : float
        Efeito Mínimo Detectável relativo desejado (ex: 0.25 para 25% de aumento relativo).
    alpha : float
        Nível de significância nominal (padrão 0.05).
    power : float
        Poder estatístico (1 - beta), padrão 0.80 (80%).
    n_variants : int
        Número total de grupos no teste (Controle + Variantes). Ex: 2 para A/B, 3 para A/B/C.
    alternative : str
        Direção do teste de hipótese:
        - 'two-sided' (padrão): Detecta diferença em qualquer direção (aumento ou queda).
        - 'one-sided': Detecta mudança apenas em uma direção específica (ex: somente aumento).

    Retorno:
    --------
    dict contendo o tamanho amostral por variante, total e parâmetros utilizados.
    """
    if alternative not in ["two-sided", "one-sided"]:
        raise ValueError("O parâmetro 'alternative' deve ser 'two-sided' ou 'one-sided'.")

    # Correção de Bonferroni para comparações múltiplas (A/B/n)
    k_comparisons = n_variants - 1
    alpha_adjusted = alpha / k_comparisons

    # Cálculo dos escores Z com base no tipo de teste
    if alternative == "two-sided":
        z_alpha = stats.norm.ppf(1 - alpha_adjusted / 2)  
    else:  
        z_alpha = stats.norm.ppf(1 - alpha_adjusted)      

    z_beta = stats.norm.ppf(power)

    # Taxa de conversão esperada no grupo tratamento
    p1 = baseline_conversion
    p2 = p1 * (1 + mde_relative)

    # Variâncias das proporções
    p_bar = (p1 + p2) / 2
    sd_null = np.sqrt(2 * p_bar * (1 - p_bar))
    sd_alt = np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))

    # Fórmula do tamanho amostral por grupo (Aproximação Normal)
    n_per_group = ((z_alpha * sd_null + z_beta * sd_alt) ** 2) / ((p2 - p1) ** 2)
    n_per_group = int(np.ceil(n_per_group))

    total_sample = n_per_group * n_variants

    return {
        "amostra_por_grupo": n_per_group,
        "amostra_total_requerida": total_sample,
        "tipo_teste": alternative,
        "alpha_nominal": alpha,
        "alpha_corrigido_bonferroni": round(alpha_adjusted, 4),
        "poder_estatistico": power,
        "baseline_rate": p1,
        "target_rate": p2,
        "mde_absoluto": round(p2 - p1, 4),
        "variantes_totais": n_variants,
    }

In [ ]:
res_unilateral = calcular_tamanho_amostra(baseline_conversion=0.08, mde_relative=0.25, alpha=0.05, power=0.80, n_variants=2, alternative="one-sided")

print("=" * 55)
print("DIMENSIONAMENTO DO TAMANHO AMOSTRAL (TESTE A/B)")
print("=" * 55)
print(f"• Tipo de Teste              : {res_unilateral['tipo_teste'].upper()} (Unilateral)")
print(f"• Taxa de Conversão Base (A) : {res_unilateral['baseline_rate']*100:.2f}%")
print(f"• Taxa Alvo Esperada (B)     : {res_unilateral['target_rate']*100:.2f}%")
print(f"• MDE Absoluto (Ganho Mínimo): +{res_unilateral['mde_absoluto']*100:.2f}% p.p.")
print("-" * 55)
print(f"• Nível de Significância (α) : {res_unilateral['alpha_nominal']*100:.1f}% (Confiança: {(1-res_unilateral['alpha_nominal'])*100:.1f}%)")
print(f"• Poder Estatístico (1-β)   : {res_unilateral['poder_estatistico']*100:.1f}%")
print(f"• Variantes Totais          : {res_unilateral['variantes_totais']} (Controle + Tratamento)")
print("=" * 55)
print(f"AMOSTRA NECESSÁRIA POR GRUPO : {res_unilateral['amostra_por_grupo']:,} usuários")
print(f"AMOSTRA TOTAL REQUERIDA     : {res_unilateral['amostra_total_requerida']:,} usuários")
print("=" * 55)

DIMENSIONAMENTO DO TAMANHO AMOSTRAL (TESTE A/B)
• Tipo de Teste              : ONE-SIDED (Unilateral)
• Taxa de Conversão Base (A) : 8.00%
• Taxa Alvo Esperada (B)     : 10.00%
• MDE Absoluto (Ganho Mínimo): +2.00% p.p.
-------------------------------------------------------
• Nível de Significância (α) : 5.0% (Confiança: 95.0%)
• Poder Estatístico (1-β)   : 80.0%
• Variantes Totais          : 2 (Controle + Tratamento)
AMOSTRA NECESSÁRIA POR GRUPO : 2,531 usuários
AMOSTRA TOTAL REQUERIDA     : 5,062 usuários


### Simulando dados do teste 

In [ ]:
np.random.seed(42)
n_samples = 10000

# Identificador do Usuário
user_ids = [f"BR-{10000 + i}" for i in range(n_samples)]

# Randomização do Grupo (50% Controle / 50% Tratamento)
groups = np.random.choice(["A", "B"], size=n_samples, p=[0.5, 0.5])

# 3. Métricas Direcionadoras (Driver / Guardrail Metrics)
# Dispositivo do usuário
devices = np.random.choice(
    ["Mobile", "Desktop", "Tablet"], size=n_samples, p=[0.6, 0.3, 0.1]
)

# Tempo na página em segundos (distribuição log-normal para simular comportamento real)
time_on_site_sec = np.round(np.random.lognormal(mean=4.2, sigma=0.6, size=n_samples), 1)

# Páginas visitadas na sessão
pages_visited = np.random.poisson(lam=4.5, size=n_samples) + 1

# 4. Probabilidade de Conversão com Efeito do Experimento (Métrica Principial)
# Base de conversão de 8% para Grupo A, com aumento relativo no Grupo B (efeito do teste)
base_conversion_prob = 0.08
effect_size = 0.02  # Grupo B converte 2% a mais (10% no total)

probs = np.where(groups == "A", base_conversion_prob, base_conversion_prob + effect_size)

# Adicionando um leve viés realista: pessoas no Desktop ou que passam mais tempo convertem mais
probs += np.where(devices == "Desktop", 0.01, 0.0)
probs = np.clip(probs, 0, 1)  # Garantir que probabilidades fiquem entre 0 e 1

converted = np.random.binomial(n=1, p=probs, size=n_samples)

# 5. Valor da Compra (Métrica de Sucesso de Negócio / Receita)
# Apenas para quem converteu (0 se não converteu)
purchase_amount = np.where(
    converted == 1,
    np.round(np.random.gamma(shape=3.0, scale=25.0, size=n_samples), 2),
    0.0,
)

# Criando o DataFrame
df_ab_test = pd.DataFrame({
        "ID": user_ids,
        "Grupo": groups,
        "Dispositivo": devices,
        "Tempo gasto no site": time_on_site_sec,
        "N° de paginas visitadas": pages_visited,
        "Converteu": converted,
        "Valor gasto": purchase_amount,
    }
)

df_ab_test.head()

,ID,Grupo,Dispositivo,Tempo gasto no site,N° de paginas visitadas,Converteu,Valor gasto
0,USR-10000,A,Mobile,41.2,4,0,0.00
1,USR-10001,B,Mobile,94.8,5,0,0.00
2,USR-10002,B,Mobile,146.0,5,1,109.82
3,USR-10003,B,Desktop,31.9,5,0,0.00
4,USR-10004,A,Mobile,186.3,6,0,0.00


In [34]:
print("--- Resumo por Grupo ---")
print(df_ab_test.groupby("Grupo").agg(total_usuarios=("ID", "count"), taxa_conversao=("Converteu", "mean"), ticket_medio=("Valor gasto", lambda x: x[x > 0].mean()), tempo_medio_site=("Tempo gasto no site", "mean")).round(2))

--- Resumo por Grupo ---
       total_usuarios  taxa_conversao  ticket_medio  tempo_medio_site
Grupo                                                                
A                5076            0.08         76.43             80.10
B                4924            0.10         75.16             79.31


### 1. Teste Z de Proporções e Intervalo de Confiança da Diferença

Calcula a significância estatística, a elevação relativa (*lift*) e o intervalo de confiança de $95\%$ para a diferença absoluta entre as taxas de conversão de dois grupos.

In [ ]:
def test_z_proporcoes(success_a: int, total_a: int, success_b: int, total_b: int, alpha: float = 0.05, alternative: str = "two-sided") -> dict:
    """Realiza o Teste Z de duas proporções independentes e calcula o IC da diferença.

    Parâmetros:
    -----------
    success_a, total_a : Sucessos e total de usuários do Grupo Controle (A).
    success_b, total_b : Sucessos e total de usuários do Grupo Tratamento (B).
    alpha : Nível de significância (padrão 0.05).
    alternative : 'two-sided' (bilateral) ou 'one-sided' (unilateral, H1: p_B > p_A).
    """
    p_a = success_a / total_a
    p_b = success_b / total_b
    diff = p_b - p_a
    lift_relativo = (diff / p_a) * 100

    # Proporção combinada para a hipótese nula
    p_pooled = (success_a + success_b) / (total_a + total_b)
    se_pooled = np.sqrt(p_pooled * (1 - p_pooled) * (1 / total_a + 1 / total_b))

    # Erro padrão não combinado para o Intervalo de Confiança
    se_diff = np.sqrt((p_a * (1 - p_a) / total_a) + (p_b * (1 - p_b) / total_b))

    # Estatística Z
    z_stat = diff / se_pooled

    # P-Valor e Escore Z Crítico
    if alternative == "two-sided":
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
        z_critical = stats.norm.ppf(1 - alpha / 2)
        ic_lower = diff - (z_critical * se_diff)
        ic_upper = diff + (z_critical * se_diff)
    elif alternative == "one-sided":
        p_value = 1 - stats.norm.cdf(z_stat)
        z_critical = stats.norm.ppf(1 - alpha)
        ic_lower = diff - (z_critical * se_diff)
        ic_upper = np.inf
    else:
        raise ValueError("'alternative' deve ser 'two-sided' ou 'one-sided'.")

    is_significant = p_value < alpha

    return {
        "taxa_conversao_A": round(p_a, 4),
        "taxa_conversao_B": round(p_b, 4),
        "diferenca_absoluta": round(diff, 4),
        "lift_relativo_pct": round(lift_relativo, 2),
        "z_stat": round(z_stat, 4),
        "p_valor": round(p_value, 5),
        "estatisticamente_significativo": is_significant,
        "ic_diferenca_95": (round(ic_lower, 4), round(ic_upper, 4)),
    }


### 2. Verificação de SRM (Sample Ratio Mismatch)

O **SRM** verifica se a distribuição dos usuários entre os grupos desviou da proporção planejada no cálculo do tamanho amostral (ex: 50/50). O teste usa o **Ajuste pelo Qui-Quadrado ($\chi^2$)**. Se $p < 0.001$, há indicativo forte de falha no pipeline de randomização ou viés de retenção.

In [35]:
def verificar_srm(observado: list[int], esperado_prop: list[float] = None, alpha: float = 0.001) -> dict:
    """Verifica a presença de Sample Ratio Mismatch (SRM) via Teste Qui-Quadrado.

    Parâmetros:
    -----------
    observado : Lista com a contagem real de usuários por grupo [n_A, n_B, ...].
    esperado_prop : Proporção esperada da alocação. Se None, assume divisão igualitária.
    alpha : Nível de corte estatístico para SRM (padrão 0.001 pela literatura de AB testing).
    """
    n_grupos = len(observado)
    total_obs = sum(observado)

    if esperado_prop is None:
        esperado_prop = [1.0 / n_grupos] * n_grupos

    esperado_counts = [p * total_obs for p in esperado_prop]

    # Teste Qui-Quadrado 
    chi2_stat, p_value = stats.chisquare(f_obs=observado, f_exp=esperado_counts)
    srm_detectado = p_value < alpha

    return {
        "contagens_observadas": observado,
        "contagens_esperadas": [round(c, 1) for c in esperado_counts],
        "chi2_stat": round(chi2_stat, 4),
        "p_valor": round(p_value, 6),
        "srm_detectado": srm_detectado,
        "status": "ALERTA: SRM detectado! Experimento inválido."
        if srm_detectado
        else "Nenhuma evidência de SRM.",
    }

### 3. Verificação de MGU (Multiple Group Unit / Vazamento de Identidade)

O **MGU** acontece quando a mesma unidade de análise (usuário/dispositivo) é exposta a **mais de um grupo** durante o experimento (vazamento entre variantes ou falha na persistência de cookies/IDs).

In [36]:
def verificar_mgu(df: pd.DataFrame, col_id: str, col_grupo: str) -> dict:
    """Detecta contaminação cruzada (MGU) identificando IDs associados a múltiplos grupos.

    Parâmetros:
    -----------
    df : DataFrame Pandas contendo os dados do experimento.
    col_id : Nome da coluna contendo o identificador do usuário.
    col_grupo : Nome da coluna contendo a variante do experimento.
    """
    # Agrupa por ID e conta variantes únicas associadas a cada ID
    variantes_por_id = df.groupby(col_id)[col_grupo].nunique()
    ids_contaminados = variantes_por_id[variantes_por_id > 1]

    total_ids_unicos = df[col_id].nunique()
    num_contaminados = len(ids_contaminados)
    taxa_mgu = (num_contaminados / total_ids_unicos) * 100

    return {
        "total_usuarios_unicos": total_ids_unicos,
        "usuarios_em_multiplos_grupos": num_contaminados,
        "taxa_mgu_pct": round(taxa_mgu, 3),
        "mgu_detectado": num_contaminados > 0,
        "status": f"ALERTA: {num_contaminados} usuário(s) afetado(s) por vazamento ({taxa_mgu:.2f}%)."
        if num_contaminados > 0
        else "Nenhum vazamento de grupo detectado entre usuários.",
    }

### Validação

In [39]:
# Checagem de MGU (Vazamento)
mgu_res = verificar_mgu(df_ab_test, col_id="ID", col_grupo="Grupo")
print("--- CHECAGEM DE MGU ---")
print(mgu_res["status"])

# Checagem de SRM (Sample Ratio Mismatch)
contagem_grupos = df_ab_test["Grupo"].value_counts().sort_index().tolist()
srm_res = verificar_srm(observado=contagem_grupos, esperado_prop=[0.5, 0.5])
print('\n')
print("--- CHECAGEM DE SRM ---")
print(srm_res["status"])

# Teste Z de Hipóteses
if not mgu_res["mgu_detectado"] and not srm_res["srm_detectado"]:
    summary = df_ab_test.groupby("Grupo")["Converteu"].agg(["sum", "count"])

    n_a, total_a = summary.loc["A", "sum"], summary.loc["A", "count"]
    n_b, total_b = summary.loc["B", "sum"], summary.loc["B", "count"]

    z_res = test_z_proporcoes(n_a, total_a, n_b, total_b, alternative="one-sided")
    print('\n')
    print("--- RESULTADO DO TESTE DE HIPÓTESES ---")
    for k, v in z_res.items():
        print(f"{k}: {v}")

--- CHECAGEM DE MGU ---
Nenhum vazamento de grupo detectado entre usuários.


--- CHECAGEM DE SRM ---
Nenhuma evidência de SRM.


--- RESULTADO DO TESTE DE HIPÓTESES ---
taxa_conversao_A: 0.0816
taxa_conversao_B: 0.1005
diferenca_absoluta: 0.019
lift_relativo_pct: 23.26
z_stat: 3.2987
p_valor: 0.00049
estatisticamente_significativo: True
ic_diferenca_95: (np.float64(0.0095), inf)


## Meta análise 

A Meta-Análise em Testes A/B combina evidências estatísticas de múltiplos experimentos, iterações ou segmentos (ex: dados históricos de múltiplos testes idênticos ou diferentes coortes regionais) para obter uma estimativa de efeito global mais precisa e com maior poder estatístico.

**Métodos Principais**

**1. Modelo de Efeitos Fixos (Fixed-Effects Model - Inverse Variance)**
* **Conceito:** Assume que existe **um único efeito real ($\theta$)** compartilhado por todos os estudos. As variações observadas entre os experimentos ocorrem puramente por erro de amostragem aleatória.
* **Ponderação:** Atribui maior peso aos estudos com **menor variância** (maior tamanho amostral).
* **Caso de uso:** Múltiplas rodadas do mesmo Teste A/B sob as mesmas condições exatas de produto/plataforma.


**2. Modelo de Efeitos Aleatórios (Random-Effects Model - DerSimonian-Laird)**
* **Conceito:** Assume que o efeito varia entre os estudos devido à heterogeneidade dos dados ($\tau^2$). O efeito observado é uma amostragem de uma **distribuição de efeitos reais**.
* **Ponderação:** Incorpora a variância dentro do estudo ($v_i$) e a variância entre estudos ($\tau^2$), equilibrando a relevância dada a estudos menores e maiores.
* **Caso de uso:** Experimentos parecidos rodados em países diferentes, plataformas distintas (iOS vs. Android) ou períodos sazonais.


**3. Teste de Heterogeneidade ($Q$ de Cochran e Estatística $I^2$)**
* **Conceito:** Medem a inconsistência entre os estudos.
* **Estatística $Q$:** Testa se as diferenças nos efeitos ultrapassam o mero acaso.
* **Estatística $I^2$:** Quantifica a porcentagem da variação total atribuível à **heterogeneidade real** ($0\%$ a $25\%$ baixa; $25\%$ a $75\%$ moderada; $>75\%$ alta). Se o $I^2$ for alto, o modelo de **Efeitos Aleatórios** deve ser o escolhido.

In [40]:
def meta_analise_efeitos_fixos(
    effects: list[float], variances: list[float], alpha: float = 0.05) -> dict:
    """Calcula a Meta-Análise de Efeitos Fixos via Método da Variância Inversa.

    Parâmetros:
    -----------
    effects : Lista de efeitos observados em cada estudo (ex: diferença de taxas de conversão p_B - p_A).
    variances : Lista das variâncias associadas a cada efeito (ex: se_diff^2).
    alpha : Nível de significância (padrão 0.05).
    """
    effects = np.array(effects)
    variances = np.array(variances)

    # Pesos inversos à variância
    weights = 1.0 / variances

    # Efeito Combinado
    pooled_effect = np.sum(weights * effects) / np.sum(weights)

    # Erro padrão e variância combinados
    pooled_var = 1.0 / np.sum(weights)
    pooled_se = np.sqrt(pooled_var)

    # Estatística Z, p-valor e IC
    z_stat = pooled_effect / pooled_se
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    z_crit = stats.norm.ppf(1 - alpha / 2)

    ic_lower = pooled_effect - z_crit * pooled_se
    ic_upper = pooled_effect + z_crit * pooled_se

    return {
        "modelo": "Efeitos Fixos",
        "efeito_combinado": round(float(pooled_effect), 5),
        "erro_padrao": round(float(pooled_se), 5),
        "z_stat": round(float(z_stat), 4),
        "p_valor": round(float(p_value), 5),
        "ic_95": (round(float(ic_lower), 5), round(float(ic_upper), 5)),
    }


def meta_analise_efeitos_aleatorios(effects: list[float], variances: list[float], alpha: float = 0.05) -> dict:
    """Calcula a Meta-Análise de Efeitos Aleatórios via Método DerSimonian-Laird.

    Incorpora a variância entre estudos (tau^2) para reponderar os efeitos.
    """
    effects = np.array(effects)
    variances = np.array(variances)
    k = len(effects)

    # 1. Pesos Fixos Iniciais
    w_fixed = 1.0 / variances
    sum_w = np.sum(w_fixed)
    sum_w_sq = np.sum(w_fixed**2)
    p_fixed = np.sum(w_fixed * effects) / sum_w

    # 2. Estatística Q de Cochran
    Q = np.sum(w_fixed * (effects - p_fixed) ** 2)

    # 3. Estatística I² (grau de heterogeneidade)
    df = k - 1
    I2 = max(0.0, ((Q - df) / Q) * 100) if Q > 0 else 0.0

    # 4. Estimativa da variância entre estudos (tau²)
    c = sum_w - (sum_w_sq / sum_w)
    tau2 = max(0.0, (Q - df) / c) if c > 0 else 0.0

    # 5. Pesos de Efeitos Aleatórios
    w_random = 1.0 / (variances + tau2)

    # 6. Efeito Combinado
    pooled_effect = np.sum(w_random * effects) / np.sum(w_random)
    pooled_var = 1.0 / np.sum(w_random)
    pooled_se = np.sqrt(pooled_var)

    # 7. Estatística Z, p-valor e IC
    z_stat = pooled_effect / pooled_se
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    z_crit = stats.norm.ppf(1 - alpha / 2)

    ic_lower = pooled_effect - z_crit * pooled_se
    ic_upper = pooled_effect + z_crit * pooled_se

    return {
        "modelo": "Efeitos Aleatórios (DerSimonian-Laird)",
        "efeito_combinado": round(float(pooled_effect), 5),
        "erro_padrao": round(float(pooled_se), 5),
        "tau2_heterogeneidade_entre_estudos": round(float(tau2), 5),
        "cochran_Q": round(float(Q), 4),
        "I2_pct": round(float(I2), 2),
        "z_stat": round(float(z_stat), 4),
        "p_valor": round(float(p_value), 5),
        "ic_95": (round(float(ic_lower), 5), round(float(ic_upper), 5)),
    }


In [24]:
# Simulando 3 testes rodados em períodos ou países diferentes:
# Efeitos observados (diferença de conversão p_B - p_A)
efeitos = [0.015, 0.022, 0.018]

# Variâncias correspondentes dos testes (se_diff^2)
variancias = [0.000025, 0.000030, 0.000020]

# Executando os dois métodos
res_fixo = meta_analise_efeitos_fixos(efeitos, variancias)
res_aleatorio = meta_analise_efeitos_aleatorios(efeitos, variancias)

print("--- META-ANÁLISE: EFEITOS FIXOS ---")
for k, v in res_fixo.items():
    print(f"{k}: {v}")

print("\n--- META-ANÁLISE: EFEITOS ALEATÓRIOS ---")
for k, v in res_aleatorio.items():
    print(f"{k}: {v}")

--- META-ANÁLISE: EFEITOS FIXOS ---
modelo: Efeitos Fixos
efeito_combinado: 0.01811
erro_padrao: 0.00285
z_stat: 6.3594
p_valor: 0.0
ic_95: (0.01253, 0.02369)

--- META-ANÁLISE: EFEITOS ALEATÓRIOS ---
modelo: Efeitos Aleatórios (DerSimonian-Laird)
efeito_combinado: 0.01811
erro_padrao: 0.00285
tau2_heterogeneidade_entre_estudos: 0.0
cochran_Q: 0.8919
I2_pct: 0.0
z_stat: 6.3594
p_valor: 0.0
ic_95: (0.01253, 0.02369)


## Conclusão 

Com base nesses resultados da Meta-Análise, a conclusão é **extremamente forte e favorável à implementação da mudança (Rollout do Grupo B)**.

### 1. Robustez do Efeito Combinado

* **Efeito Combinado de +1,81% absolutos:** A mudança no checkout gera um aumento médio consistente de **~1,81 % na taxa de conversão** através das diferentes rodadas/segmentos do teste.

* **Altíssima Significância Estatística:** O valor de $p < 0{,}0001$ e um escore $Z$ muito acima de $1{,}96$ confirmam que a chance desse ganho ser mero acaso é praticamente zero.

* **Intervalo de Confiança Positivo:** Temos $95\%$ de certeza estatística de que o ganho real na conversão está **entre +1,25% e +2,37%**. Como o limite inferior está bem acima de zero, o risco de perda é nulo.

### 2. Ausência de Heterogeneidade entre os Estudos

* O valor **$I^2 = 0{,}0\%$:** Indica **zero heterogeneidade** entre os testes. Toda a variação observada entre as rodadas é puramente ruído amostral aleatório.

* Os valores **$\tau^2 = 0{,}0$** e **$Q = 0{,}8919$:** O Teste de Cochran confirma que não há variabilidade estrutural entre os grupos.

* **Identidade entre Efeitos Fixos e Aleatórios:** O fato de ambos os modelos produzirem exatamente a mesma estimativa ($0{,}01811$ e erro padrão $0{,}00285$) reforça que o comportamento do novo checkout é altamente consistente e replicável.

### 3. Resumo para a Decisão de Negócio 

**Conclusão:** O novo fluxo de checkout apresentou um ganho médio consolidado de **+1,81% na taxa de conversão** com alta precisão e $100\%$ de consistência entre as iterações ($I^2 = 0\%$).

**Recomendação:** 

- **Aprovar o Rollout de 100% da Variante B** para toda a base de usuários. O impacto positivo é estatisticamente significante e previsível em diferentes ambientes. A simplificação do fluxo gerou um ganho consistente de **+1,81% na conversão**, com risco nulo de impacto negativo e um ROI imediato.

### 3.1 Ganho Direto na Receita 

Um aumento de **+1,81 ponto percentual** na taxa de conversão (ex: subindo de $8{,}0\%$ para $9{,}81\%$) representa uma elevação relativa de aproximadamente **+22,6% no volume total de vendas** sem que a empresa precise gastar $R\$1,00$ a mais em aquisição de tráfego.

* **Projeção de Escala:** Se o e-commerce recebe $100.000$ visitantes por mês no checkout com um Ticket Médio de $R\$ 75,00$:
* **Antes (8,0%):** $8.000$ vendas = **R$ 600.000 / mês**
* **Depois (9,81%):** $9.810$ vendas = **R$ 735.750 / mês**
* **Ganho Financeiro Limpo:** **+R$ 135.750 por mês** (+R$ 1,62 milhão/ano) alavancados puramente por otimização de conversão.

### 3.2 Retorno com Risco Zero 

Isso significa que, mesmo no pior cenário estatístico possível, o projeto **ainda entregará um ganho significativo de vendas**. Não há risco de queda na receita e o custo de implementação técnica já foi amortizado na fase de experimento.

### 3.3 Escalabilidade e Consistência do Produto

O indicador $I^2 = 0{,}0\%$ da meta-análise prova que o novo checkout funciona com **a mesma eficiência em qualquer contexto** (seja mobile/desktop, diferentes regiões ou perfis de clientes). O comportamento do consumidor respondeu de forma homogênea à redução de atrito.